# GMSPP v2 Experiments (fair-budget design)

Execution order = information value. All blocks resume from CSV.

| Block | Question | Budget | Output |
|---|---|---|---|
| C | Relaxation strength: big-M LP vs MMSP vs LP-PC | none (exact relaxation optima) | `results/v2_relax.csv` |
| B | Package (LP-PC + parallel ALNS + endgame) sweep, certified gaps | T_SWEEP each | `results/v2_alns.csv` |
| P | Pilot: BendM cold vs BendM+ (ALNS warm start) | T_MAIN each | `results/v2_pilot.csv` |
| A | Main campaign: BigM-LE / BendM / (BendM+) / package, equal budget | T_MAIN each | `results/v2_main.csv` |

Fairness protocol: every method in a table gets the SAME wall-clock
budget and thread cap. The package spends its budget as
LP-PC + workers + endgame. No speed claims across hardware.

In [ ]:
import os, sys, time, importlib
import numpy as np, pandas as pd
import multiprocessing as mp

for k in sorted(sys.modules):
    if k.startswith('gmspp'):
        importlib.reload(sys.modules[k])
from gmspp.data_structures import Instance, OBJ_MAKESPAN
from gmspp import (solve_alns_parallel, solve_benders, solve_bigm_mip_le,
                   solve_cpsat, mmsp_lower_bound,
                   generate_type1, generate_type2, load_zdf_gmspp)
from gmspp.formulation_normal import solve_normal_lp
from gmspp.formulation_bigm import solve_bigm_lp
from gmspp.alns_solver import solve_alns
from gmspp.benchmark_loader import load_and_convert_benchmark

os.makedirs('results', exist_ok=True)
print('Imports OK | CPU cores:', mp.cpu_count())

In [ ]:
# ============ CONFIG ============
RUN_C_RELAX = True     # run FIRST: cheap, budget-free
RUN_B_SWEEP = True     # package sweep, all instances
RUN_P_PILOT = False    # BendM vs BendM+ on a subset -> decide Table A row
RUN_A_MAIN  = False    # final equal-budget campaign (run LAST)

T_MAIN   = 900         # common budget for Table A / pilot (seconds)
T_SWEEP  = 300         # package sweep budget
THREADS  = 16          # thread cap for every exact method
ENDGAME_TIME = 30
ALNS_RUNS    = 10
ALNS_WORKERS = None    # None = all cores

BIGM_LP_MAX_N = 120    # skip big-M LP beyond this n (model too large)
EXACT_MAX_N   = 100    # Table A scope
OBJECTIVES = ['total_cost', 'makespan']
MS = [2, 3]

TYPE1 = [(150, 2), (150, 3), (250, 4), (400, 4)]
TYPE2 = [(150, 2), (150, 3), (250, 4), (400, 4)]
ZDF_SMALL = [('zdf1', 2), ('zdf2', 2), ('zdf3', 2), ('zdf1', 4)]
ZDF_DIR = 'benchmarks/ZDF/ZDF/ZDF'
SEED = 0

# Pilot subset: mid-size N instances where BendM struggles
PILOT_NAMES = ['N4a', 'N4c', 'N5a', 'N5c', 'N6a']

In [ ]:
# ============ INSTANCE REGISTRY ============
registry = {}
for m in MS:
    insts = load_and_convert_benchmark('benchmarks/N', m=m,
                                       cost_type='proportional')
    for name, inst in insts.items():
        for obj in OBJECTIVES:
            registry[f'N_m{m}_{obj}_{name}'] = Instance(
                items=inst.items, strips=inst.strips, objective=obj)
for n, m in TYPE1:
    registry[f'T1_{n}_{m}'] = generate_type1(n, m, seed=SEED)
for n, m in TYPE2:
    registry[f'T2_{n}_{m}'] = generate_type2(n, m, seed=SEED)
for name, m in ZDF_SMALL:
    registry[f'ZDF_{name}_{m}'] = load_zdf_gmspp(ZDF_DIR, name, m,
                                                 shuffle_seed=SEED)
print(f'{len(registry)} instances registered')

def load_done(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        return df, set(df['key'])
    return pd.DataFrame(), set()

def append_row(path, df, row):
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(path, index=False)
    return df

def dual_bound(inst, lp_tl=300):
    """max(LP-PC, MMSP, simple combinatorial bound)."""
    lp = solve_normal_lp(inst, time_limit=lp_tl)['lp_bound']
    simple = inst.simple_lower_bound()
    if inst.objective == OBJ_MAKESPAN:
        try:
            mm = mmsp_lower_bound(inst, time_limit=120)['lb']
        except Exception:
            mm = float('nan')
        cands = [lp, simple] + ([mm] if mm == mm else [])
        return max(cands), lp, mm
    return max(lp, simple), lp, float('nan')

In [ ]:
# ============ BLOCK C: RELAXATION STRENGTH ============
# big-M LP (dominated, Prop. 3) | MMSP (Vasilyev's LB) | LP-PC (ours)
if RUN_C_RELAX:
    path = 'results/v2_relax.csv'
    df, done = load_done(path)
    keys = [k for k in registry
            if registry[k].objective == OBJ_MAKESPAN and k not in done]
    print(f'C: {len(keys)} to run, {len(done)} done')
    for i, key in enumerate(keys):
        inst = registry[key]
        row = {'key': key, 'n': inst.n, 'm': inst.m}
        mm = mmsp_lower_bound(inst, time_limit=120, threads=THREADS)
        row['mmsp_lb'] = mm['lb']; row['mmsp_time'] = mm['solve_time']
        ml = mmsp_lower_bound(inst, time_limit=120, threads=THREADS,
                              relax=True)
        row['mmsp_lp'] = ml['lb']
        lp = solve_normal_lp(inst, time_limit=300)
        row['lp_pc'] = lp['lp_bound']; row['lp_pc_time'] = lp['solve_time']
        if inst.n <= BIGM_LP_MAX_N:
            bl = solve_bigm_lp(inst, time_limit=300)
            row['bigm_lp'] = bl['lp_bound']
            row['bigm_lp_time'] = bl['solve_time']
        else:
            row['bigm_lp'] = float('nan')
        row['simple_lb'] = inst.simple_lower_bound()
        row['best_lb'] = max(row['lp_pc'], row['mmsp_lb'],
                             row['simple_lb'])
        df = append_row(path, df, row)
        print(f"[{i+1}/{len(keys)}] {key}: bigM={row['bigm_lp']:.1f} "
              f"MMSP-LP={row['mmsp_lp']:.1f} "
              f"MMSP={row['mmsp_lb']:.1f} LP-PC={row['lp_pc']:.1f}")

In [ ]:
# ============ BLOCK B: PACKAGE SWEEP (certified gaps) ============
if RUN_B_SWEEP:
    path = 'results/v2_alns.csv'
    df, done = load_done(path)
    keys = [k for k in registry if k not in done]
    print(f'B: {len(keys)} to run at {T_SWEEP}s, {len(done)} done')
    for i, key in enumerate(keys):
        inst = registry[key]
        lb, lp, mm = dual_bound(inst)
        r = solve_alns_parallel(inst, n_runs=ALNS_RUNS,
                                n_workers=ALNS_WORKERS,
                                time_limit=T_SWEEP, compute_dual=False,
                                endgame_time=ENDGAME_TIME, verbose=False)
        gap = (r.objective - lb) / max(r.objective, 1e-9) * 100 if lb > 0 else float('nan')
        row = {'key': key, 'n': inst.n, 'm': inst.m,
               'objective': inst.objective,
               'obj': r.objective, 'lb': lb, 'lp_pc': lp, 'mmsp': mm,
               'gap_pct': gap, 'time': r.total_time,
               'obj_after_alns': r.obj_after_alns,
               'ls_moves': r.ls_moves, 'endgame_strips': r.endgame_strips,
               'obj_mean': r.obj_mean, 'obj_std': r.obj_std}
        df = append_row(path, df, row)
        print(f"[{i+1}/{len(keys)}] {key}: obj={r.objective:.0f} "
              f"gap={gap:.1f}%")

In [ ]:
# ============ BLOCK P: PILOT (BendM vs BendM+) ============
if RUN_P_PILOT:
    path = 'results/v2_pilot.csv'
    df, done = load_done(path)
    keys = [f'N_m{m}_{obj}_{nm}' for m in MS for obj in OBJECTIVES
            for nm in PILOT_NAMES]
    keys = [k for k in keys if k in registry and k not in done]
    print(f'P: {len(keys)} to run at {T_MAIN}s each x 2 methods')
    for i, key in enumerate(keys):
        inst = registry[key]
        row = {'key': key, 'n': inst.n, 'm': inst.m,
               'objective': inst.objective}
        # cold
        r0 = solve_benders(inst, time_limit=T_MAIN, threads=THREADS,
                           use_skyline_heuristic=False, verbose=False)
        row.update(cold_obj=r0.objective, cold_lb=r0.lower_bound,
                   cold_gap=r0.gap_percent, cold_opt=r0.optimal)
        # warm: LP-PC/MMSP + 60s ALNS + BendM remainder
        t0 = time.time()
        lb, _, _ = dual_bound(inst, lp_tl=60)
        a = solve_alns(inst, time_limit=60, seed=42, verbose=False,
                       ls_time=10, endgame_time=15)
        rem = max(T_MAIN - (time.time() - t0), 10)
        r1 = solve_benders(inst, time_limit=rem, threads=THREADS,
                           use_skyline_heuristic=False,
                           mip_start=a.solution, mip_start_obj=a.objective,
                           lower_bound_inject=lb, verbose=False)
        row.update(warm_alns=a.objective,
                   warm_obj=min(r1.objective, a.objective),
                   warm_lb=r1.lower_bound, warm_gap=r1.gap_percent,
                   warm_opt=r1.optimal)
        df = append_row(path, df, row)
        print(f"[{i+1}/{len(keys)}] {key}: cold={r0.objective:.0f}"
              f"/{r0.gap_percent:.1f}% warm={row['warm_obj']:.0f}"
              f"/{r1.gap_percent:.1f}%")

In [ ]:
# ============ BLOCK A: MAIN CAMPAIGN (equal budget T_MAIN) ============
# Methods: bigmle | bendm | bendm_plus (if pilot positive) | package
METHODS_A = ['bigmle', 'bendm', 'package']   # add 'bendm_plus' after pilot
if RUN_A_MAIN:
    path = 'results/v2_main.csv'
    df, done0 = load_done(path)
    done = set(zip(df['key'], df['method'])) if len(df) else set()
    keys = [k for k in registry
            if k.startswith('N_') and registry[k].n <= EXACT_MAX_N]
    todo = [(k, meth) for k in keys for meth in METHODS_A
            if (k, meth) not in done]
    print(f'A: {len(todo)} (instance, method) pairs at {T_MAIN}s')
    for i, (key, meth) in enumerate(todo):
        inst = registry[key]
        row = {'key': key, 'n': inst.n, 'm': inst.m,
               'objective': inst.objective, 'method': meth}
        t0 = time.time()
        if meth == 'bigmle':
            r = solve_bigm_mip_le(inst, time_limit=T_MAIN, threads=THREADS,
                                  lp_pc_time_limit=120)
            row.update(obj=r['objective'], lb=r['lower_bound'],
                       gap=r['gap_pct'], opt=r['optimal'])
        elif meth == 'bendm':
            r = solve_benders(inst, time_limit=T_MAIN, threads=THREADS,
                              use_skyline_heuristic=False)
            row.update(obj=r.objective, lb=r.lower_bound,
                       gap=r.gap_percent, opt=r.optimal)
        elif meth == 'bendm_plus':
            lb0, _, _ = dual_bound(inst, lp_tl=60)
            a = solve_alns(inst, time_limit=60, seed=42, verbose=False,
                           ls_time=10, endgame_time=15)
            rem = max(T_MAIN - (time.time() - t0), 10)
            r = solve_benders(inst, time_limit=rem, threads=THREADS,
                              use_skyline_heuristic=False,
                              mip_start=a.solution,
                              mip_start_obj=a.objective,
                              lower_bound_inject=lb0)
            row.update(obj=min(r.objective, a.objective),
                       lb=r.lower_bound, gap=r.gap_percent, opt=r.optimal)
        elif meth == 'package':
            lb0, _, _ = dual_bound(inst)
            r = solve_alns_parallel(inst, n_runs=ALNS_RUNS,
                                    n_workers=ALNS_WORKERS,
                                    time_limit=T_MAIN - 60,
                                    compute_dual=False,
                                    endgame_time=ENDGAME_TIME,
                                    verbose=False)
            g = ((r.objective - lb0) / max(r.objective, 1e-9) * 100
                 if lb0 > 0 else float('nan'))
            row.update(obj=r.objective, lb=lb0, gap=g,
                       opt=abs(g) < 1e-6)
        row['time'] = time.time() - t0
        df = append_row(path, df, row)
        print(f"[{i+1}/{len(todo)}] {key} {meth}: obj={row['obj']:.0f} "
              f"gap={row['gap']:.1f}%")

## Analysis

In [ ]:
# Tier 3: relaxation strength (Table C)
if os.path.exists('results/v2_relax.csv'):
    rx = pd.read_csv('results/v2_relax.csv')
    rx['family'] = rx['key'].str.split('_').str[0]
    rx['lp_vs_mmsp'] = (rx['lp_pc'] - rx['mmsp_lb']) / rx['mmsp_lb'].clip(lower=1e-9) * 100
    print('=== Table C: LP-PC vs MMSP (% stronger) by family ===')
    print(rx.groupby('family')['lp_vs_mmsp']
            .agg(['count', 'mean', 'min', 'max']).round(1))
    both = rx.dropna(subset=['bigm_lp'])
    if len(both):
        v3 = (both['bigm_lp'] > both['lp_pc'] + 1e-6).sum()
        print(f'Prop-3 check (bigM-LP <= LP-PC): {v3} violations/{len(both)}')
    if 'mmsp_lp' in rx.columns:
        v4 = (rx['mmsp_lp'] > rx['lp_pc'] + 1e-6).sum()
        print(f'Prop-4 check (MMSP-LP <= LP-PC): {v4} violations/{len(rx)}')
        rx['geom_gain'] = rx['lp_pc'] - rx['mmsp_lp']
        rx['int_gain'] = rx['mmsp_lb'] - rx['mmsp_lp']
        print('increments over the liquified base (mean by family):')
        print(rx.groupby('family')[['geom_gain', 'int_gain']]
                .mean().round(2))

# Tier 1: certified gaps (Block B)
if os.path.exists('results/v2_alns.csv'):
    al = pd.read_csv('results/v2_alns.csv')
    print()
    print('=== Package certified gaps (vs max(LP-PC, MMSP)) ===')
    print(al.groupby(['objective'])['gap_pct']
            .agg(['count', 'mean', 'median', 'max']).round(2))

# Pilot verdict
if os.path.exists('results/v2_pilot.csv'):
    pl = pd.read_csv('results/v2_pilot.csv')
    print()
    print('=== Pilot: BendM+ vs BendM ===')
    pl['d_obj'] = (pl['warm_obj'] - pl['cold_obj']) / pl['cold_obj'] * 100
    pl['d_gap'] = pl['warm_gap'] - pl['cold_gap']
    print(pl.groupby('objective')[['d_obj', 'd_gap']].mean().round(2))
    print('(negative = warm better; decide METHODS_A accordingly)')

# Table A summary
if os.path.exists('results/v2_main.csv'):
    mn = pd.read_csv('results/v2_main.csv')
    print()
    print('=== Table A: equal-budget comparison ===')
    print(mn.groupby(['objective', 'method'])
            .agg(obj=('obj', 'mean'), gap=('gap', 'mean'),
                 opt=('opt', 'sum')).round(2))